In [1]:
import os
import time
from pymmcore_plus import CMMCorePlus
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QScrollArea, QTabWidget, QFileDialog,
    QRadioButton, QFrame, QSpinBox, QLineEdit, QCheckBox, QPushButton,
    QLabel, QHBoxLayout, QVBoxLayout, QComboBox, QWidget, QTableWidget,
    QTableWidgetItem, QMessageBox, QInputDialog, QGridLayout, QSizePolicy,
    QGroupBox, QSplitter
)
from PyQt5.QtGui import QPixmap, QImage, QFont
from PyQt5.QtCore import QTimer, pyqtSignal, pyqtSlot, Qt, QThread, QSize
from PyQt5 import QtGui, QtCore, QtWidgets
import serial
import numpy as np
import pandas as pd
import sys
from PIL import Image
import csv
import cv2
from skimage import exposure
from collections import defaultdict
from matplotlib.figure import Figure
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
import copy
import tifffile as tiff
import torch
import shutil
from glob import glob
from sam2.build_sam import build_sam2_video_predictor

# ─────────────────────────────────────────────
#  Hardware constants
# ─────────────────────────────────────────────
Arduino_port      = "COM4"
Arduino_baud_rate = 115200
Arduino_timeout   = 0.1

Relay_port    = 'COM6'
Relay_baudrate = 19200
Relay_timeout  = 0.1


# ─────────────────────────────────────────────
#  Style helpers
# ─────────────────────────────────────────────
BTN_RED    = "background-color: #bb283a; color: white; border-radius: 4px; padding: 4px 8px;"
BTN_GREEN  = "background-color: #2ac555; color: white; border-radius: 4px; padding: 4px 8px;"
BTN_BLUE   = "background-color: #3a6bc9; color: white; border-radius: 4px; padding: 4px 8px;"
BTN_YELLOW = "background-color: #d4a017; color: black; border-radius: 4px; padding: 4px 8px;"
BTN_BLACK  = "background-color: #222222; color: white; border-radius: 4px; padding: 4px 8px;"
BTN_PURPLE = "background-color: #6a3db8; color: white; border-radius: 4px; padding: 4px 8px;"

LABEL_BOLD = "font-weight: bold; color: #cccccc;"
GROUP_STYLE = """
QGroupBox {
    border: 1px solid #555555;
    border-radius: 6px;
    margin-top: 10px;
    padding-top: 4px;
    color: #aaaaaa;
    font-weight: bold;
}
QGroupBox::title {
    subcontrol-origin: margin;
    left: 8px;
    padding: 0 4px;
}
"""

WIDGET_STYLE = """
    QTableWidget, QTableView {
        background-color: #1e1e1e;
        color: #ffffff;
        gridline-color: #444444;
        border: 1px solid #444444;
    }
    QTableWidget::item { color: #ffffff; background-color: #1e1e1e; }
    QTableWidget::item:selected { background-color: #2a82da; color: white; }
    QHeaderView::section {
        background-color: #2d2d2d;
        color: #cccccc;
        border: 1px solid #444444;
        padding: 3px;
    }
    QComboBox {
        background-color: #2d2d2d;
        color: #ffffff;
        border: 1px solid #555555;
        border-radius: 4px;
        padding: 3px 6px;
    }
    QComboBox QAbstractItemView {
        background-color: #2d2d2d;
        color: #ffffff;
        selection-background-color: #2a82da;
    }
    QComboBox::drop-down { border: none; }
    QSpinBox, QDoubleSpinBox {
        background-color: #2d2d2d;
        color: #ffffff;
        border: 1px solid #555555;
        border-radius: 4px;
        padding: 3px 6px;
    }
    QSpinBox::up-button, QSpinBox::down-button,
    QDoubleSpinBox::up-button, QDoubleSpinBox::down-button {
        background-color: #3a3a3a;
        border: none;
    }
    QLineEdit {
        background-color: #2d2d2d;
        color: #ffffff;
        border: 1px solid #555555;
        border-radius: 4px;
        padding: 3px 6px;
    }
    QScrollBar:vertical, QScrollBar:horizontal {
        background-color: #1e1e1e;
        border: none;
    }
    QScrollBar::handle:vertical, QScrollBar::handle:horizontal {
        background-color: #555555;
        border-radius: 3px;
        min-height: 20px;
    }
    QCheckBox { color: #cccccc; }
    QLabel { color: #cccccc; }
"""


DARK_PALETTE = {
    "Window":          (53,  53,  53),
    "WindowText":      (255, 255, 255),
    "Base":            (25,  25,  25),
    "AlternateBase":   (53,  53,  53),
    "ToolTipBase":     (255, 255, 255),
    "ToolTipText":     (255, 255, 255),
    "Text":            (255, 255, 255),
    "Button":          (53,  53,  53),
    "ButtonText":      (255, 255, 255),
    "BrightText":      (255, 0,   0),
    "Link":            (42,  130, 218),
    "Highlight":       (42,  130, 218),
    "HighlightedText": (0,   0,   0),
}


def make_dark_palette():
    p = QtGui.QPalette()
    mapping = {
        "Window":          QtGui.QPalette.Window,
        "WindowText":      QtGui.QPalette.WindowText,
        "Base":            QtGui.QPalette.Base,
        "AlternateBase":   QtGui.QPalette.AlternateBase,
        "ToolTipBase":     QtGui.QPalette.ToolTipBase,
        "ToolTipText":     QtGui.QPalette.ToolTipText,
        "Text":            QtGui.QPalette.Text,
        "Button":          QtGui.QPalette.Button,
        "ButtonText":      QtGui.QPalette.ButtonText,
        "BrightText":      QtGui.QPalette.BrightText,
        "Link":            QtGui.QPalette.Link,
        "Highlight":       QtGui.QPalette.Highlight,
        "HighlightedText": QtGui.QPalette.HighlightedText,
    }
    for name, role in mapping.items():
        p.setColor(role, QtGui.QColor(*DARK_PALETTE[name]))
    return p


def group(title, layout, flat=False):
    """Wrap a layout in a styled QGroupBox."""
    box = QGroupBox(title)
    box.setStyleSheet(GROUP_STYLE)
    box.setFlat(flat)
    box.setLayout(layout)
    return box


def hline():
    line = QFrame()
    line.setFrameShape(QFrame.HLine)
    line.setFrameShadow(QFrame.Sunken)
    line.setStyleSheet("color: #444444;")
    return line


def labeled_input(label_text, widget, label_width=None):
    """Return an HBoxLayout with a label + widget."""
    lbl = QLabel(label_text)
    lbl.setStyleSheet("color: #aaaaaa;")
    if label_width:
        lbl.setFixedWidth(label_width)
    row = QHBoxLayout()
    row.addWidget(lbl)
    row.addWidget(widget)
    return row


# ─────────────────────────────────────────────
#  Hardware init
# ─────────────────────────────────────────────
def initialize_arduino(port, baud_rate, timeout):
    try:
        arduino = serial.Serial(port, baud_rate, timeout=timeout)
        time.sleep(2)
        print(f"Connected to {port} at {baud_rate} baud.")
        return arduino
    except serial.SerialException:
        print(f"Error: Could not open serial port {port}.")
        return None

def handshake(arduino):
    if arduino:
        for _ in range(5):
            arduino.write(b"HELLO\n")
            time.sleep(0.5)
            response = arduino.readline().decode('utf-8').strip()
            if response == "READY":
                print("Handshake successful!")
                return True
        print("Handshake failed!")
        return False
    return False

arduino = initialize_arduino(Arduino_port, Arduino_baud_rate, Arduino_timeout)
if arduino and handshake(arduino):
    print("Serial connection established.")
else:
    print("Failed to establish a connection. Exiting.")
    if arduino:
        arduino.close()
    exit()

def send_to_arduino(data):
    arduino.write(f"{data}\n".encode())
    time.sleep(0.1)
    response = arduino.readline().decode().strip()
    print(f"Arduino Response: {response}" if response else "No response received.")

def set_voltage(set_value):
    if str(set_value) == 'High':
        send_to_arduino(5)
    elif str(set_value) == 'Low':
        send_to_arduino(4)

    print(f"Voltage set to: {str(set_value)}")


def init_serial_port(relayport, relay_Baudrate, relay_timeout):
    try:
        obj = serial.Serial(relayport, relay_Baudrate, timeout=relay_timeout)
        obj.write(b'')
        obj.flush()
        obj.write_terminator = b'\r'
        print("Numato relay correctly connected")
        return obj
    except serial.SerialException:
        print("Numato relay NOT correctly connected")
        sys.exit(1)

Numato_device = init_serial_port(Relay_port, Relay_baudrate, Relay_timeout)

# ═══════════════════════════════════════════════════════════════════════════
#  SAM2Config  –  edit paths / thresholds before use
# ═══════════════════════════════════════════════════════════════════════════

class SAM2Config:
    CHECKPOINT     = "C:\SAM2/sam2.1_hiera_small.pt"
    MODEL_CONFIG   = "C:\SAM2/sam2.1_hiera_s.yaml"
    DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"
    OBJ_ID         = 1
    MIN_COMP_AREA  = 200    # px² – discard mask fragments smaller than this
    AREA_THRESHOLD = 0.70   # trigger chemostat when area < 70 % of initial
    BF_EXPOSURE_MS = 20     # ms – brightfield snap exposure


# ═══════════════════════════════════════════════════════════════════════════
#  SAM2Manager  –  all SAM2 logic, completely decoupled from the GUI
# ═══════════════════════════════════════════════════════════════════════════

class SAM2Manager:
    """
    Owns the SAM2 video predictor and all per-position tracking state.

    Per-position state (dicts keyed by 0-based position index):
        ref_image_path  – path to the initial BF TIFF used during annotation
        ref_mask        – cleaned boolean ndarray produced by annotation
        initial_area    – droplet area (px) recorded from ref_mask
        loop_bf_paths   – ordered list of BF TIFFs snapped in the *current*
                          loop; reset to [] after each trigger
        pos_loop_count  – number of completed loops (0 = never triggered,
                          so currently in "Loop 1")
    """

    def __init__(self):
        self.predictor      = None   # loaded lazily on first use
        self.base_dir       = "."

        self.ref_image_path = {}   # {idx: str}
        self.ref_mask       = {}   # {idx: np.ndarray bool}
        self.initial_area   = {}   # {idx: int px}
        self.loop_bf_paths  = {}   # {idx: list[str]}
        self.pos_loop_count = {}   # {idx: int}

    # ── predictor ──────────────────────────────────────────────────────────
    def load_predictor(self):
        if self.predictor is None:
            print("[SAM2] Loading predictor …")
            self.predictor = build_sam2_video_predictor(
                SAM2Config.MODEL_CONFIG,
                SAM2Config.CHECKPOINT,
                device=SAM2Config.DEVICE,
            )
            print("[SAM2] Predictor ready.")

    # ── image helpers ───────────────────────────────────────────────────────
    @staticmethod
    def load_gray_uint8(path: str) -> np.ndarray:
        """Read any TIFF/PNG → normalised uint8 grayscale 2-D array."""
        img = tiff.imread(path)
        if img.ndim > 2:
            img = img[..., 0]
        img = img.astype(np.float32)
        return cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    @staticmethod
    def write_sam2_frame_folder(image_paths: list, folder: str) -> str:
        """
        Write a numbered JPG sequence into `folder` that SAM2 expects.
        The folder is wiped and rebuilt each call so frame indices always
        match the supplied image_paths list exactly.
        """
        if os.path.exists(folder):
            shutil.rmtree(folder)
        os.makedirs(folder)
        for i, p in enumerate(image_paths):
            gray = SAM2Manager.load_gray_uint8(p)
            rgb  = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
            cv2.imwrite(
                os.path.join(folder, f"{i:05d}.jpg"),
                cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR),
            )
        return folder

    # ── mask post-processing ────────────────────────────────────────────────
    def clean_mask(self, mask: np.ndarray) -> np.ndarray:
        """
        1. Remove connected components smaller than MIN_COMP_AREA.
        2. Bridge electrode-split regions with a convex hull.
        3. Morphological closing to smooth edges.
        Returns a boolean array.
        """
        u8 = mask.astype(np.uint8)
        n, labels, stats, _ = cv2.connectedComponentsWithStats(u8, 8)
        cleaned = np.zeros_like(u8)
        for i in range(1, n):
            if stats[i, cv2.CC_STAT_AREA] >= SAM2Config.MIN_COMP_AREA:
                cleaned[labels == i] = 1

        contours, _ = cv2.findContours(
            cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if not contours:
            return cleaned.astype(bool)
        hull   = cv2.convexHull(np.vstack(contours))
        filled = np.zeros_like(cleaned)
        cv2.drawContours(filled, [hull], -1, 1, -1)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        filled = cv2.morphologyEx(filled, cv2.MORPH_CLOSE, kernel)
        return filled.astype(bool)

    # ── interactive annotation ──────────────────────────────────────────────
    def annotate_frame_interactive(
        self, gray: np.ndarray, inference_state
    ) -> np.ndarray:
        """
        Blocking matplotlib window for point-click annotation.
          Left-click   = positive point (droplet interior)
          Middle-click = negative point (background / electrode)
          Right-click  = undo last point
          Enter        = accept mask and close

        Returns the cleaned boolean mask.
        Raises RuntimeError if closed without any annotation.
        """
        import matplotlib
        matplotlib.use("QtAgg")
        import matplotlib.pyplot as plt

        predictor  = self.predictor
        points     = []
        labels     = []
        cur_mask   = [None]
        mask_art   = [None]
        pt_arts    = []

        fig, ax = plt.subplots(figsize=(9, 9))
        ax.imshow(gray, cmap="gray")
        ax.set_title(
            "Left = droplet  |  Middle = background  |  "
            "Right = undo  |  Enter = accept"
        )

        def _redraw_points():
            for a in pt_arts:
                a.remove()
            pt_arts.clear()
            for (x, y), lab in zip(points, labels):
                pt_arts.append(
                    ax.plot(x, y, "go" if lab == 1 else "ro", markersize=7)[0]
                )

        def _run_sam2():
            if not points:
                return
            pt_arr = np.array(points, dtype=np.float32)
            lb_arr = np.array(labels, dtype=np.int32)
            ctx = (
                torch.autocast("cuda", dtype=torch.bfloat16)
                if SAM2Config.DEVICE == "cuda"
                else torch.autocast("cpu", enabled=False)
            )
            with torch.inference_mode(), ctx:
                _, _, logits = predictor.add_new_points_or_box(
                    inference_state=inference_state,
                    frame_idx=0,
                    obj_id=SAM2Config.OBJ_ID,
                    points=pt_arr,
                    labels=lb_arr,
                )
            raw = (logits[0] > 0).cpu().numpy()
            cur_mask[0] = np.squeeze(raw).astype(bool)

            if mask_art[0] is not None:
                mask_art[0].remove()
            ov = np.zeros((*cur_mask[0].shape, 4), dtype=np.float32)
            ov[cur_mask[0]] = [1.0, 0.35, 0.0, 0.45]
            mask_art[0] = ax.imshow(ov)
            _redraw_points()
            fig.canvas.draw_idle()

        def _onclick(ev):
            if ev.inaxes != ax or ev.xdata is None:
                return
            x, y = float(ev.xdata), float(ev.ydata)
            if ev.button == 1:
                points.append([x, y]); labels.append(1)
            elif ev.button == 2:
                points.append([x, y]); labels.append(0)
            elif ev.button == 3:
                if not points:
                    return
                points.pop(); labels.pop()
                if not points:
                    if mask_art[0] is not None:
                        mask_art[0].remove(); mask_art[0] = None
                    _redraw_points(); fig.canvas.draw_idle(); return
            _run_sam2()

        def _onkey(ev):
            if ev.key == "enter":
                plt.close(fig)

        fig.canvas.mpl_connect("button_press_event", _onclick)
        fig.canvas.mpl_connect("key_press_event",    _onkey)
        plt.show(block=True)

        if cur_mask[0] is None:
            raise RuntimeError(
                "No annotation given – add at least one positive point."
            )
        return self.clean_mask(cur_mask[0])

    # ── SAM2 propagation helpers ────────────────────────────────────────────
    def _autocast_ctx(self):
        if SAM2Config.DEVICE == "cuda":
            return torch.autocast("cuda", dtype=torch.bfloat16)
        return torch.autocast("cpu", enabled=False)

    def _seed_state_with_mask(self, state, frame_idx: int, mask: np.ndarray):
        """
        Seed SAM2 at `frame_idx` using the centroid of `mask` as a single
        positive point.  This injects the reference mask as prior context.
        """
        ys, xs = np.where(mask)
        if len(xs) == 0:
            raise RuntimeError("Reference mask is empty – cannot seed SAM2.")
        cx, cy = float(xs.mean()), float(ys.mean())
        with torch.inference_mode(), self._autocast_ctx():
            self.predictor.add_new_points_or_box(
                inference_state=state,
                frame_idx=frame_idx,
                obj_id=SAM2Config.OBJ_ID,
                points=np.array([[cx, cy]], dtype=np.float32),
                labels=np.array([1],         dtype=np.int32),
            )

    def measure_area_with_context(
        self, pos_idx: int, new_bf_path: str
    ) -> tuple:
        """
        Measure the droplet area in `new_bf_path` using a SAM2 inference
        context built from:

            frame 0          →  initial BF  (seeded with ref_mask centroid)
            frames 1 … N     →  all BF images already in current loop
            frame N+1        →  new_bf_path  (the frame we want to predict)

        The initial frame is always frame 0 and always seeded with the
        reference mask, regardless of how many loop frames have accumulated.

        Returns (cleaned_mask: np.ndarray[bool], area_px: int).
        """
        self.load_predictor()

        ref_img   = self.ref_image_path[pos_idx]
        ref_mask  = self.ref_mask[pos_idx]
        loop_imgs = self.loop_bf_paths.get(pos_idx, [])

        # Full ordered frame list  [initial] + [loop so far] + [new frame]
        all_frames      = [ref_img] + loop_imgs + [new_bf_path]
        query_frame_idx = len(all_frames) - 1

        # Write the numbered JPG folder SAM2 needs
        tmp_dir = os.path.join(self.base_dir, f"_sam2_tmp_pos{pos_idx}")
        self.write_sam2_frame_folder(all_frames, tmp_dir)

        state = self.predictor.init_state(video_path=tmp_dir)

        # Seed frame 0 (initial BF) with the reference mask centroid
        self._seed_state_with_mask(state, frame_idx=0, mask=ref_mask)

        # Propagate forward; keep only the prediction at the query frame
        final_mask = None
        with torch.inference_mode(), self._autocast_ctx():
            for out_idx, _, out_logits in self.predictor.propagate_in_video(
                state,
                start_frame_idx=0,
                max_frame_num_to_track=len(all_frames),
            ):
                if out_idx == query_frame_idx:
                    raw        = (out_logits[0] > 0).cpu().numpy()
                    final_mask = np.squeeze(raw).astype(bool)

        if final_mask is None:
            return np.zeros_like(ref_mask, dtype=bool), 0

        cleaned = self.clean_mask(final_mask)
        return cleaned, int(cleaned.sum())

    # ── context management ──────────────────────────────────────────────────
    def append_loop_bf(self, pos_idx: int, path: str):
        """Add a newly captured BF to this position's current-loop list."""
        self.loop_bf_paths.setdefault(pos_idx, []).append(path)

    def reset_loop_context(self, pos_idx: int):
        """
        Called after a trigger.  Discards all current-loop BF images so the
        next loop starts with only the initial frame as context.
        """
        self.loop_bf_paths[pos_idx] = []



Microscope = {}
Microscope['mmc'] = CMMCorePlus.instance()
Microscope['mmc'].loadSystemConfiguration("C:\\MATLAB Microscope\\AmoghMMConfig_Hamamatsu.cfg")


# ─────────────────────────────────────────────
#  Custom toggle button
# ─────────────────────────────────────────────
class QToggleButton(QPushButton):
    def __init__(self, text='', parent=None):
        super().__init__(text, parent)
        self.setCheckable(True)
        self.setChecked(False)
        self.setStyleSheet(BTN_RED)


# ─────────────────────────────────────────────
#  Video thread
# ─────────────────────────────────────────────
class VideoThread(QThread):
    change_pixmap_signal = pyqtSignal(np.ndarray)

    def __init__(self):
        super().__init__()
        self._run_flag   = True
        self._record_flag = False
        self.out = None
        self.video_directory = "C:/Users/Cell Culture Scope/Downloads/Videos"
        self.video_filename  = self.get_unique_filename(self.video_directory)
        self.fourcc     = cv2.VideoWriter_fourcc(*'MJPG')
        self.fps        = 10
        self.frame_size = None
        self.mmc        = Microscope['mmc']
        self.camera     = self.mmc.getCameraDevice()
        self.DIAshutter = 'TIDiaShutter'
        self.focus      = self.mmc.getFocusDevice()
        self.stage      = self.mmc.getXYStageDevice()
        self.PFS        = self.mmc.getAutoFocusDevice()
        self.EPIshutter = 'TIEpiShutter'
        self.DIAlamp    = 'TIDiaLamp'
        self.scope      = 'TIScope'
        self.zoom       = 'TINosePiece'
        self.filter     = 'TIFilterBlock1'
        self.lightpath  = 'TILightPath'
        self.PFS_offset = 'TIPFSOffset'
        self.core       = 'Core'
        self.camerapath = '2-Left100'
        self.mmc.setProperty(self.camera, "CONVERSION FACTOR COEFF", "0.5")

    def run(self):
        self.mmc.setProperty(self.DIAshutter, 'State', 0)
        self.mmc.setProperty(self.EPIshutter,  'State', 0)
        self.mmc.setProperty(self.core, 'Shutter', self.DIAshutter)
        self.mmc.setProperty(self.lightpath, 'Label', self.camerapath)
        self.mmc.initializeCircularBuffer()
        self.mmc.prepareSequenceAcquisition(self.camera)
        self.mmc.waitForDevice(self.DIAshutter)
        self.mmc.waitForDevice(self.camera)
        self.mmc.startContinuousSequenceAcquisition(100)
        self.Sequencing = self.mmc.isSequenceRunning()
        while self._run_flag and self.Sequencing:
            if self.mmc.getRemainingImageCount() > 0:
                liveimage    = self.mmc.getLastImage()
                image_width  = self.mmc.getImageWidth()
                image_height = self.mmc.getImageHeight()
                self.final_image = self.convert_raw_np(liveimage, image_width, image_height, np.uint16)
                self.change_pixmap_signal.emit(self.final_image)
                if self._record_flag:
                    if self.out is None:
                        self.video_filename = self.get_unique_filename(self.video_directory)
                        self.frame_size = (image_width, image_height)
                        self.out = cv2.VideoWriter(self.video_filename, self.fourcc, self.fps,
                                                   self.frame_size, isColor=True)
                    if self.out is not None and self.out.isOpened():
                        frame_rescaled = exposure.rescale_intensity(
                            self.final_image, in_range='image', out_range='uint8').astype(np.uint8)
                        frame_to_save = cv2.cvtColor(frame_rescaled, cv2.COLOR_GRAY2BGR)
                        try:
                            self.out.write(frame_to_save)
                        except Exception as e:
                            print(f"Exception during video write: {e}")
                            self._record_flag = False
                            if self.out:
                                self.out.release(); self.out = None
        self.mmc.stopSequenceAcquisition(self.camera)
        self.mmc.clearCircularBuffer()
        if self.out:
            self.out.release()

    def convert_raw_np(self, raw_img, img_width, img_height, pixel_Type):
        rawImage = np.frombuffer(raw_img, dtype=pixel_Type).reshape((img_height, img_width)).T
        return exposure.rescale_intensity(rawImage)

    def start_recording(self):
        self._record_flag = True

    def stop_recording(self):
        self._record_flag = False
        if self.out:
            self.out.release(); self.out = None

    def get_unique_filename(self, directory, base="output", ext=".avi"):
        os.makedirs(directory, exist_ok=True)
        i, filename = 1, os.path.join(directory, f"{base}{ext}")
        while os.path.exists(filename):
            filename = os.path.join(directory, f"{base}_{i}{ext}"); i += 1
        return filename

    def stop(self):
        self._record_flag = False
        if self.out:
            self.out.release(); self.out = None
        self._run_flag = False
        self.wait()


# ─────────────────────────────────────────────
#  Droplet worker thread
# ─────────────────────────────────────────────
class DropletWorker(QThread):
    def __init__(self, operation, input_1, input_2=None,
                 purge_duration=0, flow_duration=0, drive_duration=0,
                 chemostat_number=4, PWM_duration1=0.05, PWM_duration2=0.05,
                 PWM_totalduration=5):
        super().__init__()
        self.Numato_port      = Numato_device
        self.operation        = operation
        self.input_1          = input_1
        self.input_2          = input_2
        self.chemostat        = chemostat_number
        self.Purge_duration   = purge_duration
        self.Flow_duration    = flow_duration
        self.Drive_duration   = drive_duration
        self.PWM_duration1    = PWM_duration1
        self.PWM_duration2    = PWM_duration2
        self.PWM_totalduration = PWM_totalduration

    def run(self):
        ops = {
            "purge":      lambda: self.purge_inlet(self.input_1, self.input_2),
            "generate":   lambda: self.generate_droplet(self.input_1, self.input_2),
            "drive":      lambda: self.drive_droplet(self.chemostat),
            "characterize": lambda: self.characterize_droplet(self.chemostat),
            "wash":       lambda: self.wash_step(self.input_1, self.input_2),
        
        }

        ops.get(self.operation, lambda: print("Invalid operation"))()

    # ── valve helpers ──────────────────────────────────────
    def send_relay_command(self, command):
        if self.Numato_port and self.Numato_port.is_open:
            try:
                self.Numato_port.write(f"{command}\r".encode('utf-8'))
                time.sleep(0.005)
            except serial.SerialException:
                print('Worker thread failed to communicate with device')

    def get_relay_id(self, idx):
        return str(idx) if idx <= 9 else chr(ord('A') + (idx - 10))

    def control_valve(self, idx, state):
        relay_id = self.get_relay_id(idx)
        self.send_relay_command(f"relay {'off' if state else 'on'} {relay_id}")

    # ── operations ────────────────────────────────────────
    def characterize_droplet(self, chemostat_number):
        for v in [8, 9, 10, 11]: self.control_valve(v, state=False)
        time.sleep(3)
        self.control_valve(15, state=False); self.control_valve(7, state=False)
        time.sleep(self.Purge_duration)
        self.control_valve(2, state=True);  self.control_valve(5, state=True)
        time.sleep(0.5)
        self.control_valve(7, state=True);  self.control_valve(4, state=False)
        self.control_valve(3, state=False)
        time.sleep(self.Flow_duration)
        self.control_valve(3, state=True);  self.control_valve(4, state=True)
        time.sleep(2)
        self.control_valve(15, state=True)
        self.control_valve(2, state=False); self.control_valve(5, state=False)
        self.control_valve(chemostat_number + 7, state=True)
        send_to_arduino(2)
        time.sleep(self.Drive_duration)
        self.control_valve(chemostat_number + 7, state=False)
        send_to_arduino(3)
        self.control_valve(4, state=False)
        time.sleep(1)
        self.control_valve(4, state=True)

    def purge_inlet(self, input_1, input_2):
        self.control_valve(input_1 + 11, state=False)
        time.sleep(1)
        self.control_valve(7, state=False)
        self.control_valve(input_2 + 11, state=False)
        time.sleep(self.Purge_duration)
        self.control_valve(input_1 + 11, state=True)
        self.control_valve(input_2 + 11, state=True)
        self.control_valve(7, state=True)
        time.sleep(20)

    def generate_droplet(self, input_1, input_2):
        self.control_valve(2, state=True);  self.control_valve(5, state=True)
        self.control_valve(15, state=False); time.sleep(0.1)
        self.control_valve(3, state=False); self.control_valve(4, state=False)
        time.sleep(self.Flow_duration)
        self.control_valve(3, state=True);  self.control_valve(4, state=True)
        self.control_valve(15, state=True)

    def drive_droplet(self, Chemostat_number):
        self.control_valve(2, state=False); self.control_valve(5, state=False)
        self.control_valve(Chemostat_number + 7, state=True)
        send_to_arduino(2)
        time.sleep(self.Drive_duration)
        send_to_arduino(3)
        self.control_valve(Chemostat_number + 7, state=False)
        self.control_valve(4, state=False)
        time.sleep(3)
        self.control_valve(4, state=True)

    def wash_step(self, input_1, input_2):
        self.control_valve(input_1 + 11, state=False)
        self.control_valve(input_2 + 11, state=False)
        time.sleep(4)
        self.control_valve(input_2 + 11, state=True)
        self.control_valve(7, state=False)
        time.sleep(5)
        self.control_valve(input_1 + 11, state=True)
        self.control_valve(7, state=True)



# ═══════════════════════════════════════════════════════════
#  MAIN GUI
# ═══════════════════════════════════════════════════════════
class MicroscopeControlGUI(QMainWindow):
    def __init__(self):
        super().__init__()
        self.Numato_port  = Numato_device
        self.video_thread = None
        self.positions    = []
        self.sam2_mgr = SAM2Manager()
        self.selected_exposures = []
        self.Chemostat_protocol_steps = []
        self.current_protocol_table_step = 0
        self._init_microscope()
        self._build_ui()
        self.setWindowTitle('Microscope Control')
        self.resize(1400, 860)
        self.show()

    # ── microscope init ───────────────────────────────────
    def _init_microscope(self):
        self.mmc = Microscope['mmc']
        self.camera     = self.mmc.getCameraDevice()
        self.DIAshutter = self.mmc.getShutterDevice()
        self.focus      = self.mmc.getFocusDevice()
        self.stage      = self.mmc.getXYStageDevice()
        self.PFS        = self.mmc.getAutoFocusDevice()
        self.EPIshutter = 'TIEpiShutter'
        self.DIAlamp    = 'TIDiaLamp'
        self.scope      = 'TIScope'
        self.zoom       = 'TINosePiece'
        self.filter     = 'TIFilterBlock1'
        self.lightpath  = 'TILightPath'
        self.PFS_offset = 'TIPFSOffset'
        self.core       = 'Core'
        self.eyepath    = '1-Eye100'
        self.camerapath = '2-Left100'
        self.zoom4x  = '1-(Achromat) 4x NA 0.10 Dry'
        self.zoom10x = '2-(Achromat) 10x NA 0.25 Dry'
        self.zoom20x = '3-(Achromat) 20x NA 0.40 Dry'
        self.zoom40x = '4-S Plan Fluor 40x NA 0.60 Dry'
        self.zoom60x = '5-Plan Apo 60x NA 1.40 Oil'
        self.zoomempty = '6-Unknown'
        self.filterNames = {1:'1- FITC', 2:'2- DAPI', 3:'3- BFP-A',
                            4:'4- Cy5',  5:'5- Cy3',  6:'6- DIA'}
        # camera setup
        self.mmc.setProperty(self.camera, 'Sensor Cooler', 'ON')
        self.mmc.setProperty(self.camera, 'Exposure', 20)
        self.mmc.setProperty(self.camera, 'MINIMUM ACQUISITION TIMEOUT', 500)
        self.mmc.setProperty(self.camera, 'Binning', '4x4')
        self.mmc.setProperty(self.camera, 'ScanMode', 1)
        self.mmc.setProperty(self.camera, 'CONVERSION FACTOR COEFF', 0.5)
        self.mmc.setProperty(self.camera, 'PixelType', '16bit')
        self.mmc.setProperty(self.DIAlamp, 'ComputerControl', 'On')
        self.mmc.setProperty(self.DIAlamp, 'Intensity', 4)
        self.mmc.setProperty(self.DIAlamp, 'State', 0)
        self.mmc.setProperty(self.DIAshutter, 'State', 0)
        self.mmc.setProperty(self.EPIshutter,  'State', 0)
        self.mmc.setProperty(self.lightpath, 'Label', self.eyepath)
        os.chdir('C:/Users/Cell Culture Scope/Documents/MATLAB')

    # ══════════════════════════════════════════
    #  UI builder
    # ══════════════════════════════════════════
    def _build_ui(self):
        self.tabs = QTabWidget()
        self.tabs.setStyleSheet("""
            QTabWidget::pane { border: 1px solid #444; }
            QTabBar::tab { background: #333; color: #aaa; padding: 6px 18px; border-radius: 4px 4px 0 0; }
            QTabBar::tab:selected { background: #555; color: white; }
        """)
        self.setCentralWidget(self.tabs)

        tab1 = QWidget(); self.tabs.addTab(tab1, "Microscope Control")
        tab2 = QWidget(); self.tabs.addTab(tab2, "Timelapse Setup")
        sam2_widget = QWidget()
        sam2_layout = QVBoxLayout(sam2_widget)
        self._build_sam2_panel(sam2_layout)
        self.tabs.addTab(sam2_widget, "SAM2 Tracking")

        self._build_tab1(tab1)
        self._build_tab2(tab2)

    # ══════════════════════════════════════════
    #  TAB 1 — Microscope Control
    # ══════════════════════════════════════════
    def _build_tab1(self, parent):
        root = QHBoxLayout(parent)
        root.setSpacing(10)
        root.setContentsMargins(10, 10, 10, 10)

        # ── LEFT PANEL ─────────────────────────
        left = QVBoxLayout()
        left.setSpacing(8)

        # -- Illumination group
        illum_grid = QGridLayout()
        illum_grid.setSpacing(6)
        self.dialamponlight = QToggleButton("DIA Lamp")
        self.dialamponlight.clicked.connect(self.DIAlamp_ON)
        self.DIAshutterbutton = QToggleButton("DIA Shutter")
        self.DIAshutterbutton.clicked.connect(self.toggle_DIA_shutter)
        self.EPIshutterbutton = QToggleButton("EPI Shutter")
        self.EPIshutterbutton.clicked.connect(self.toggle_EPI_shutter)
        illum_grid.addWidget(self.dialamponlight,    0, 0)
        illum_grid.addWidget(self.DIAshutterbutton,  0, 1)
        illum_grid.addWidget(self.EPIshutterbutton,  0, 2)
        left.addWidget(group("Illumination", illum_grid))

        # -- Objective + light path
        obj_layout = QHBoxLayout()
        obj_layout.setSpacing(6)
        self.Zoom_list = QComboBox()
        self.Zoom_list.addItems([self.zoom4x, self.zoom10x, self.zoom20x,
                                  self.zoom40x, self.zoom60x, self.zoomempty])
        self.Zoom_list.currentTextChanged.connect(self.Set_zoom)
        self.eyepathlight    = QPushButton("→ Eye")
        self.camerapathlight = QPushButton("→ Camera")
        for btn in (self.eyepathlight, self.camerapathlight):
            btn.setStyleSheet(BTN_BLUE)
            btn.clicked.connect(self.PathtoCamera)
        obj_layout.addWidget(self.Zoom_list, 2)
        obj_layout.addWidget(self.camerapathlight, 1)
        obj_layout.addWidget(self.eyepathlight,    1)
        left.addWidget(group("Objective & Light Path", obj_layout))

        # -- Filter buttons
        filter_grid = QGridLayout()
        filter_grid.setSpacing(4)
        for fk, fv in self.filterNames.items():
            btn = QPushButton(fv)
            btn.setStyleSheet(BTN_PURPLE)
            btn.clicked.connect(lambda checked, fn=fk: self.change_filter(fn))
            r, c = divmod(fk - 1, 3)
            filter_grid.addWidget(btn, r, c)
        left.addWidget(group("Filters", filter_grid))

        # -- Image viewer (fixed, never collapses)
        self.image_Live = QLabel()
        self.image_Live.setAlignment(Qt.AlignCenter)
        self.image_Live.setStyleSheet("background-color: #111111; border-radius: 4px;")
        self.image_Live.setMinimumSize(340, 260)
        self.image_Live.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.image_Live.setText("No image loaded")

        # -- Image capture buttons (always above the viewer)
        cap_layout = QHBoxLayout()
        cap_layout.setSpacing(6)
        self.snap_Button   = QPushButton("Snap Image")
        self.live_Button   = QToggleButton("Live Image")
        self.save_Button   = QPushButton("Save Image")
        self.record_button = QPushButton("Start Recording")
        self.record_button.setCheckable(True)
        for btn, style in [(self.snap_Button,   BTN_BLUE),
                           (self.live_Button,   BTN_RED),
                           (self.save_Button,   BTN_BLUE),
                           (self.record_button, BTN_RED)]:
            btn.setStyleSheet(style)
        self.snap_Button.clicked.connect(self.snap_DIA_image)
        self.live_Button.clicked.connect(self.startstoplive_imaging)
        self.save_Button.clicked.connect(self.save_image)
        self.record_button.clicked.connect(self.handle_record_button)
        for btn in (self.snap_Button, self.live_Button, self.save_Button, self.record_button):
            cap_layout.addWidget(btn)

        viewer_layout = QVBoxLayout()
        viewer_layout.setSpacing(4)
        viewer_layout.addLayout(cap_layout)
        viewer_layout.addWidget(self.image_Live, 1)
        left.addWidget(group("Image Viewer", viewer_layout), 1)

        ''' # -- Stage controls
        stage_grid = QGridLayout()
        stage_grid.setSpacing(4)
        self.stagefast, self.stagemedium, self.stageslow = 1000, 100, 10
        self.stage_speed = self.stagefast
        self.stagespeedbutton = QComboBox()
        self.stagespeedbutton.addItems([str(self.stageslow), str(self.stagemedium), str(self.stagefast)])
        self.stagespeedbutton.setCurrentIndex(2)
        self.stagespeedbutton.currentTextChanged.connect(self.Set_stage_speed)
        self.Xplus  = QPushButton("X+"); self.Xminus = QPushButton("X−")
        self.Yplus  = QPushButton("Y+"); self.Yminus = QPushButton("Y−")
        self.Zplus  = QPushButton("Z+"); self.Zminus = QPushButton("Z−")
        for btn in (self.Xplus, self.Xminus, self.Yplus, self.Yminus, self.Zplus, self.Zminus):
            btn.setStyleSheet(BTN_BLUE)
        self.Xplus.clicked.connect( lambda: self.mmc.setXYPosition(self.mmc.getXPosition(self.stage)+self.stage_speed, self.mmc.getYPosition(self.stage)))
        self.Xminus.clicked.connect(lambda: self.mmc.setXYPosition(self.mmc.getXPosition(self.stage)-self.stage_speed, self.mmc.getYPosition(self.stage)))
        self.Yplus.clicked.connect( lambda: self.mmc.setXYPosition(self.mmc.getXPosition(self.stage), self.mmc.getYPosition(self.stage)+self.stage_speed))
        self.Yminus.clicked.connect(lambda: self.mmc.setXYPosition(self.mmc.getXPosition(self.stage), self.mmc.getYPosition(self.stage)-self.stage_speed))
        self.Zplus.clicked.connect( lambda: self.mmc.setPosition(self.mmc.getPosition()+self.stage_speed))
        self.Zminus.clicked.connect(lambda: self.mmc.setPosition(self.mmc.getPosition()-self.stage_speed))
        speed_lbl = QLabel("Step (µm):"); speed_lbl.setStyleSheet("color:#aaa;")
        stage_grid.addWidget(self.Xplus,  0, 0); stage_grid.addWidget(self.Xminus, 0, 1)
        stage_grid.addWidget(self.Yplus,  1, 0); stage_grid.addWidget(self.Yminus, 1, 1)
        stage_grid.addWidget(self.Zplus,  2, 0); stage_grid.addWidget(self.Zminus, 2, 1)
        stage_grid.addWidget(speed_lbl,   3, 0); stage_grid.addWidget(self.stagespeedbutton, 3, 1)
        left.addWidget(group("Stage", stage_grid)) '''

        # ── RIGHT PANEL ────────────────────────
        right = QVBoxLayout()
        right.setSpacing(8)

        # -- Saved positions
        pos_layout = QVBoxLayout()
        pos_layout.setSpacing(4)
        self.Positions_table = QTableWidget(self)
        self.Positions_table.setRowCount(8)
        self.Positions_table.setColumnCount(3)
        self.Positions_table.setHorizontalHeaderLabels(['X', 'Y', 'Z'])
        self.Positions_table.verticalHeader().setDefaultSectionSize(22)
        self.Positions_table.setMaximumHeight(220)
        #self.Positions_table.horizontalHeader().setStretchLastSection(True)
        pos_btns = QHBoxLayout()
        pos_btns.setSpacing(4)
        self.save_Position_button  = QPushButton("Add Position")
        self.replacePositionButton = QPushButton("Replace")
        self.clearButton           = QPushButton("Clear All")
        self.GoToPositionButton = QPushButton("Go to Position:")
        for btn, style in [(self.save_Position_button,  BTN_GREEN),
                           (self.replacePositionButton, BTN_BLUE),
                           (self.GoToPositionButton, BTN_BLACK),
                           (self.clearButton, BTN_RED)]:
            btn.setStyleSheet(style)
        
        self.save_Position_button.clicked.connect(self.save_Position)
        #self.replacePositionButton.clicked.connect(self.replacePosition)
        self.Positions_spinbox  = QSpinBox(); self.Positions_spinbox.setRange(1, 8); self.Positions_spinbox.setValue(1)
        self.replacePositionButton.clicked.connect(lambda: self.replacePosition(self.Positions_spinbox.value()))
        self.GoToPositionButton.clicked.connect(lambda: self.GoToPosition(self.Positions_spinbox.value()))

        self.clearButton.clicked.connect(self.clearPositions)
        for btn in (self.GoToPositionButton, self.Positions_spinbox, self.replacePositionButton, self.save_Position_button, self.clearButton):
            pos_btns.addWidget(btn)
        pos_layout.addLayout(pos_btns)
        pos_layout.addWidget(self.Positions_table)
        right.addWidget(group("Saved Positions (max 8)", pos_layout))

        # -- Valve grid
        valve_grid = QGridLayout()
        valve_grid.setSpacing(4)
        self.controls = []
        for i in range(21):
            btn = QPushButton(f"V{i+1}")
            btn.setCheckable(True)
            btn.setStyleSheet(BTN_RED)
            btn.setFixedHeight(28)
            btn.clicked.connect(lambda state, idx=i: self.control_valve(idx, state))
            self.controls.append(btn)
            valve_grid.addWidget(btn, i // 7, i % 7)
        valve_btns = QHBoxLayout()
        self.stop_all_button = QPushButton("Stop All")
        self.stop_all_button.setStyleSheet(BTN_BLACK)
        self.all_on_button   = QPushButton("All On")
        self.all_on_button.setStyleSheet(BTN_BLUE)
        self.stop_all_button.clicked.connect(self.stop_all_callback)
        self.all_on_button.clicked.connect(self.all_on_callback)
        valve_btns.addWidget(self.stop_all_button)
        valve_btns.addWidget(self.all_on_button)
        full_valve = QVBoxLayout()
        full_valve.setSpacing(4)
        full_valve.addLayout(valve_grid)
        full_valve.addLayout(valve_btns)
        right.addWidget(group("Valve Controls", full_valve))

        # -- Droplet parameters
        dp_grid = QGridLayout()
        dp_grid.setSpacing(4)
        self.purge_duration_Input  = QLineEdit("0")
        self.flow_duration_Input   = QLineEdit("0")
        self.drive_duration_Input  = QLineEdit("0")
        self.inlet_Input           = QLineEdit("1")
        for row, (lbl, widget) in enumerate([
            ("Purge duration (s):",       self.purge_duration_Input),
            ("Aqueous flow duration (s):", self.flow_duration_Input),
            ("Drive duration (s):",        self.drive_duration_Input),
            ("Chemostat number:",          self.inlet_Input),
        ]):
            l = QLabel(lbl); l.setStyleSheet("color:#aaa;")
            dp_grid.addWidget(l, row, 0)
            dp_grid.addWidget(widget, row, 1)
        self.Generate_drop_button = QPushButton("Generate Droplet")
        self.Generate_drop_button.setStyleSheet(BTN_GREEN)
        self.Generate_drop_button.clicked.connect(lambda: self.Characterize_Droplet(input=3))
        dp_grid.addWidget(self.Generate_drop_button, 4, 0, 1, 2)
        right.addWidget(group("Droplet Parameters", dp_grid))

        # -- Voltage
        volt_layout = QVBoxLayout(); volt_layout.setSpacing(6)
        volt_row = QHBoxLayout()
        v_lbl = QLabel("Operating voltage (V):"); v_lbl.setStyleSheet("color:#aaa;")
        self.voltagevalues = QComboBox()
        self.voltagevalues.addItems(['High', 'Low'])
        self.voltagevalues.currentTextChanged.connect(set_voltage)
        volt_row.addWidget(v_lbl); volt_row.addWidget(self.voltagevalues)
        self.TurnOnVolts = QPushButton("Voltage Signal")
        self.TurnOnVolts.setCheckable(True)
        self.TurnOnVolts.setStyleSheet(BTN_RED)
        self.TurnOnVolts.clicked.connect(self.alter_Arduino_state)
        volt_layout.addLayout(volt_row)
        volt_layout.addWidget(self.TurnOnVolts)
        right.addWidget(group("Voltage", volt_layout))
        
        right.addStretch()

        root.addLayout(left, 3)
        root.addLayout(right, 2)

    # ══════════════════════════════════════════
    #  TAB 2 — Timelapse Setup
    # ══════════════════════════════════════════
    def _build_tab2(self, parent):
        root = QHBoxLayout(parent)
        root.setSpacing(10)
        root.setContentsMargins(10, 10, 10, 10)

        # ── LEFT COLUMN ────────────────────────
        left = QVBoxLayout()
        left.setSpacing(8)

        # -- Filter / exposure table
        fe_layout = QVBoxLayout()
        fe_layout.setSpacing(4)
        fe_top = QHBoxLayout()
        filter_list_layout = QVBoxLayout()
        filter_list_layout.setSpacing(2)
        for fk, fv in self.filterNames.items():
            lbl = QLabel(fv); lbl.setStyleSheet("color:#aaa; font-size:11px;")
            filter_list_layout.addWidget(lbl)
        self.Exposures_table = QTableWidget(len(self.filterNames), 2)
        self.Exposures_table.setHorizontalHeaderLabels(["Filter (1-6)", "Exposure (ms)"])
        self.Exposures_table.verticalHeader().setDefaultSectionSize(24)
        self.Exposures_table.setMaximumHeight(180)
        for row in range(len(self.filterNames)):
            sb1 = QSpinBox(); sb1.setRange(1, 6); sb1.setValue(row + 1)
            sb2 = QSpinBox(); sb2.setRange(0, 1000); sb2.setValue(0)
            self.Exposures_table.setCellWidget(row, 0, sb1)
            self.Exposures_table.setCellWidget(row, 1, sb2)
        self.Exposures_table.horizontalHeader().setStretchLastSection(True)
        fe_top.addLayout(filter_list_layout)
        fe_top.addWidget(self.Exposures_table, 1)
        fe_bottom = QHBoxLayout()
        self.Save_Exposures_button = QPushButton("Save Values")
        self.Save_Exposures_button.setStyleSheet(BTN_GREEN)
        self.Save_Exposures_button.clicked.connect(self.read_Exposure_values)
        self.quick_EPI_filter   = QSpinBox(); self.quick_EPI_filter.setRange(1,6); self.quick_EPI_filter.setValue(4)
        self.quick_EPI_exposure = QLineEdit("100")
        self.snap_EPI_button    = QPushButton("Snap Fluorescent Image")
        self.snap_EPI_button.setStyleSheet(BTN_BLUE)
        self.snap_EPI_button.clicked.connect(
            lambda: self.snap_EPI_image(self.quick_EPI_filter.value(), int(self.quick_EPI_exposure.text())))
        quick_lbl_f = QLabel("Filter:"); quick_lbl_f.setStyleSheet("color:#aaa;")
        quick_lbl_e = QLabel("Exposure:");quick_lbl_e.setStyleSheet("color:#aaa;")
        fe_bottom.addWidget(self.Save_Exposures_button)
        fe_bottom.addWidget(quick_lbl_f)
        fe_bottom.addWidget(self.quick_EPI_filter)
        fe_bottom.addWidget(quick_lbl_e)
        fe_bottom.addWidget(self.quick_EPI_exposure)
        fe_bottom.addWidget(self.snap_EPI_button)
        fe_layout.addLayout(fe_top)
        fe_layout.addLayout(fe_bottom)
        left.addWidget(group("Fluorescence Imaging", fe_layout))

        # -- Protocol definition
        proto_top = QHBoxLayout()
        proto_top.setSpacing(10)

        # inputs spinboxes
        inp_layout = QGridLayout(); inp_layout.setSpacing(4)
        self.Chemostat_inputs = []
        for i in range(4):
            lbl = QLabel(f"Input {i+1}:"); lbl.setStyleSheet("color:#aaa;")
            sb  = QSpinBox(); sb.setRange(0, 5); sb.setValue(0)
            inp_layout.addWidget(lbl, i, 0)
            inp_layout.addWidget(sb,  i, 1)
            self.Chemostat_inputs.append(sb)
        proto_top.addWidget(group("Inputs", inp_layout))

        # chemostat checkboxes
        chem_layout = QVBoxLayout(); chem_layout.setSpacing(2)
        self.Chemostats = []
        for i in range(8):
            cb = QCheckBox(f"Chemostat {i+1}"); cb.setStyleSheet("color:#ccc;")
            chem_layout.addWidget(cb); self.Chemostats.append(cb)
        proto_top.addWidget(group("Chemostats", chem_layout))

        # step buttons
        step_btn_layout = QVBoxLayout(); step_btn_layout.setSpacing(6)
        self.add_Step_button    = QPushButton("Add Step");        self.add_Step_button.setStyleSheet(BTN_GREEN)
        self.clear_last_button  = QPushButton("Clear Last Step"); self.clear_last_button.setStyleSheet(BTN_BLUE)
        self.clear_Steps_button = QPushButton("Clear All Steps"); self.clear_Steps_button.setStyleSheet(BTN_RED)
        self.export_button      = QPushButton("Export to CSV");   self.export_button.setStyleSheet(BTN_BLACK)
        self.add_Step_button.clicked.connect(self.add_loading_step)
        self.clear_last_button.clicked.connect(self.clear_last_step)
        self.clear_Steps_button.clicked.connect(self.clear_all_steps)
        self.export_button.clicked.connect(self.export_to_csv)
        for btn in (self.add_Step_button, self.clear_last_button, self.clear_Steps_button, self.export_button):
            step_btn_layout.addWidget(btn)
        step_btn_layout.addStretch()
        proto_top.addWidget(group("Actions", step_btn_layout))
        left.addWidget(group("Protocol Definition", proto_top))

        # -- Protocol table (scrollable)
        self.Chemostat_protocol_table = QTableWidget()
        self.Chemostat_protocol_table.setRowCount(10)
        self.Chemostat_protocol_table.setColumnCount(8)
        self.Chemostat_protocol_table.setVerticalHeaderLabels(
            ["Input 1", "Input 2"] + [f"Chemostat {i+1}" for i in range(8)])
        for c in range(8):
            self.Chemostat_protocol_table.setHorizontalHeaderItem(c, QTableWidgetItem(f"Step {c+1}"))
        for c in range(8):
            for r in range(10):
                if r < 2:
                    self.Chemostat_protocol_table.setItem(r, c, QTableWidgetItem(""))
                else:
                    self.Chemostat_protocol_table.setCellWidget(r, c, self.create_centered_checkbox())
        self.Chemostat_protocol_table.setStyleSheet("QTableWidget { gridline-color: #444; }")
        self.Chemostat_protocol_table.verticalHeader().setDefaultSectionSize(24)
        proto_tbl_layout = QVBoxLayout()
        proto_tbl_layout.addWidget(self.Chemostat_protocol_table)
        left.addWidget(group("Protocol Steps", proto_tbl_layout))
        left.addStretch()

        # ── RIGHT COLUMN ───────────────────────
        right = QVBoxLayout()
        right.setSpacing(8)

        # -- Experiment timing
        timing_grid = QGridLayout(); timing_grid.setSpacing(6)
        self.cycle_Interval_Input = QLineEdit("0")
        self.cycle_Input          = QLineEdit("0")
        for r, (lbl, w) in enumerate([
            ("Cycle interval (min):",      self.cycle_Interval_Input),
            ("Total duration (min):",      self.cycle_Input)
        ]):
            l = QLabel(lbl); l.setStyleSheet("color:#aaa;")
            timing_grid.addWidget(l, r, 0)
            timing_grid.addWidget(w, r, 1)
        self.interval = 0; self.cycles = 0
        right.addWidget(group("Timing", timing_grid))

        # -- Directory
        dir_layout = QHBoxLayout(); dir_layout.setSpacing(4)
        self.directory_input = QLineEdit()
        self.directory_input.setPlaceholderText("Save directory…")
        self.browse_button   = QPushButton("Browse")
        self.browse_button.setStyleSheet(BTN_BLUE)
        self.browse_button.clicked.connect(self.browse_folder)
        dir_layout.addWidget(self.directory_input, 1)
        dir_layout.addWidget(self.browse_button)
        right.addWidget(group("Save Directory", dir_layout))

        

        # -- Experiment control
        exp_layout = QVBoxLayout(); exp_layout.setSpacing(6)
        self.start_experiment_button = QPushButton("▶  Start Experiment")
        self.stop_experiment_button  = QPushButton("■  Stop Experiment")
        self.start_experiment_button.setStyleSheet(BTN_GREEN)
        self.stop_experiment_button.setStyleSheet(BTN_RED)
        self.start_experiment_button.setMinimumHeight(36)
        self.stop_experiment_button.setMinimumHeight(36)
        self.start_experiment_button.clicked.connect(self._on_start_experiment)
        exp_layout.addWidget(self.start_experiment_button)
        exp_layout.addWidget(self.stop_experiment_button)
        right.addWidget(group("Experiment Control", exp_layout))

        right.addStretch()

        root.addLayout(left,  3)
        root.addLayout(right, 1)

    # ══════════════════════════════════════════
    #  Microscope slots
    # ══════════════════════════════════════════
    def startstoplive_imaging(self):
        if self.live_Button.isChecked():
            self.live_Button.setStyleSheet(BTN_GREEN)
            self.video_thread = VideoThread()
            self.video_thread.change_pixmap_signal.connect(self.update_image)
            self.video_thread.start()
        else:
            if self.video_thread:
                self.video_thread.stop()
            self.live_Button.setStyleSheet(BTN_RED)

    def handle_record_button(self):
        if self.record_button.isChecked():
            self.record_button.setText("● Recording")
            self.record_button.setStyleSheet(BTN_GREEN)
            if self.video_thread:
                self.video_thread.start_recording()
        else:
            self.record_button.setText("Start Recording")
            self.record_button.setStyleSheet(BTN_RED)
            if self.video_thread:
                self.video_thread.stop_recording()

    def snap_DIA_image(self):
        self.image_path = "C:\\Users\\Cell Culture Scope\\Pictures\\image.png"
        self.mmc.setProperty(self.core, 'Shutter', self.DIAshutter)
        self.mmc.waitForSystem()
        self.mmc.setExposure(self.camera, 10)
        self.mmc.setProperty(self.camera, 'CONVERSION FACTOR COEFF', 0.5)
        self.mmc.setProperty(self.lightpath, 'Label', self.camerapath)
        self.mmc.waitForDevice(self.filter)
        self.mmc.waitForSystem()
        self.mmc.setAutoShutter(True)
        self.mmc.snapImage()
        rawImage    = self.mmc.getImage()
        image_width = self.mmc.getImageWidth()
        image_height = self.mmc.getImageHeight()

        rawImage = np.frombuffer(rawImage, dtype=np.uint16).reshape((image_height, image_width)).T
        self.adjusted_image = exposure.rescale_intensity(rawImage)
        raw_data = self.adjusted_image.tobytes()
        self.myQImage = QImage(raw_data, image_width, image_height, QImage.Format_Grayscale16)
        self.image_Live.setPixmap(QPixmap.fromImage(self.myQImage).scaled(
            self.image_Live.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))
        return rawImage      # ← add this one line

    def snap_EPI_image(self, Filter, Exposure_value):
        self.image_path = "C:\\Users\\Cell Culture Scope\\Pictures\\image.png"
        self.mmc.setProperty(self.core, 'Shutter', self.EPIshutter)
        self.mmc.waitForSystem()
        self.change_filter(Filter)
        self.mmc.setExposure(self.camera, Exposure_value)
        self.mmc.setProperty(self.lightpath, 'Label', self.camerapath)
        self.mmc.waitForDevice(self.filter)
        self.mmc.waitForSystem()
        self.mmc.snapImage()
        rawImage = self.mmc.getImage()
        image_width  = self.mmc.getImageWidth()
        image_height = self.mmc.getImageHeight()

        rawImage = np.frombuffer(rawImage, dtype=np.uint16).reshape((image_height, image_width)).T
        self.adjusted_image = exposure.rescale_intensity(rawImage)
        raw_data_rescaled = self.adjusted_image.tobytes()
        self.myQImage = QImage(raw_data_rescaled, image_width, image_height, QImage.Format_Grayscale16)
        self.myQImage.save('C:\\Users\\Cell Culture Scope\\Pictures\\image_Qimage.png')
        self.image_Live.setPixmap(QPixmap.fromImage(self.myQImage).scaled(
            self.image_Live.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))
        return rawImage    # ← raw uint16, not self.adjusted_image

    def save_image(self, save_path=None):
        if self.image_Live.pixmap():
            self.image_Live.pixmap().save(self.image_path)
            print(f"Image saved to {self.image_path}")
        else:
            print("No image to save")



    @pyqtSlot(np.ndarray)
    def update_image(self, cv_img):
        h, w = cv_img.shape
        raw_data = cv_img.tobytes()
        qimg = QImage(raw_data, w, h, QImage.Format_Grayscale16)
        self.image_Live.setPixmap(QPixmap.fromImage(qimg).scaled(
            self.image_Live.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))

    def DIAlamp_ON(self):
        if self.dialamponlight.isChecked():
            self.mmc.setProperty(self.DIAlamp, 'State', 1)
            self.dialamponlight.setStyleSheet(BTN_GREEN)
        else:
            self.mmc.setProperty(self.DIAlamp, 'State', 0)
            self.dialamponlight.setStyleSheet(BTN_RED)
    
    def DIALamp_activate(self):
        self.mmc.setProperty(self.DIAlamp, 'State', 1)
        time.sleep(1)

    def DIALamp_deactivate(self):
        self.mmc.setProperty(self.DIAlamp, 'State', 0)
        time.sleep(1)

    def save_Position(self):
        if len(self.positions) < 8:
            self.positions.append(list(self.get_new_position()))
            self.updateTable()
        else:
            QMessageBox.warning(self, 'Limit Reached', 'Cannot save more than 8 positions.')

    def get_new_position(self):
        return (self.mmc.getXPosition(self.stage),
                self.mmc.getYPosition(self.stage),
                self.mmc.getPosition())

    def updateTable(self):
        for row, (x, y, z) in enumerate(self.positions):
            self.Positions_table.setItem(row, 0, QTableWidgetItem(f'{x:.2f}'))
            self.Positions_table.setItem(row, 1, QTableWidgetItem(f'{y:.2f}'))
            self.Positions_table.setItem(row, 2, QTableWidgetItem(f'{z:.2f}'))
        for row in range(len(self.positions), 8):
            for col in range(3):
                self.Positions_table.setItem(row, col, QTableWidgetItem(''))

    def clearPositions(self):
        self.positions = []
        self.updateTable()

    def replacePosition(self, position_number):
        if not self.positions:
            QMessageBox.warning(self, 'No Positions', 'No positions available to replace.')
            return
        
        idx = position_number - 1
        if idx < len(self.positions):
            self.positions[idx] = list(self.get_new_position())
            self.updateTable()
    
    def GoToPosition(self, position_number):
        
        idx = position_number - 1
        if idx < len(self.positions):
            self.mmc.setXYPosition(self.positions[idx][0], self.positions[idx][1])
            self.mmc.setPosition(self.positions[idx][2])
            self.mmc.waitForSystem(); time.sleep(0.5)

    def PathtoCamera(self):
        self.mmc.setProperty(self.lightpath, 'Label', self.camerapath)

    def Set_zoom(self, item):
        self.mmc.setProperty(self.zoom, 'Label', item)

    def change_filter(self, filter_name):
        self.mmc.setProperty(self.filter, 'State', filter_name - 1)

    def Set_stage_speed(self, item):
        self.stage_speed = int(item)

    def toggle_DIA_shutter(self):
        if self.DIAshutterbutton.isChecked():
            self.mmc.setProperty(self.DIAshutter, 'State', 1)
            self.DIAshutterbutton.setStyleSheet(BTN_GREEN)
        else:
            self.mmc.setProperty(self.DIAshutter, 'State', 0)
            self.DIAshutterbutton.setStyleSheet(BTN_RED)

    def toggle_EPI_shutter(self):
        if self.EPIshutterbutton.isChecked():
            self.mmc.setProperty(self.EPIshutter, 'State', 1)
            self.EPIshutterbutton.setStyleSheet(BTN_GREEN)
        else:
            self.mmc.setProperty(self.EPIshutter, 'State', 0)
            self.EPIshutterbutton.setStyleSheet(BTN_RED)

    # ── valve slots ───────────────────────────
    def get_relay_id(self, idx):
        return str(idx) if idx <= 9 else chr(ord('A') + (idx - 10))

    def control_valve(self, idx, state):
        relay_id = self.get_relay_id(idx)
        if state:
            self.controls[idx].setStyleSheet(BTN_GREEN)
            self.send_relay_command(f"relay off {relay_id}")
        else:
            self.controls[idx].setStyleSheet(BTN_RED)
            self.send_relay_command(f"relay on {relay_id}")

    def stop_all_callback(self):
        for i, ctrl in enumerate(self.controls):
            ctrl.setStyleSheet(BTN_RED); ctrl.setChecked(False)
            self.send_relay_command(f"relay on {self.get_relay_id(i)}")
        self.send_relay_command('open all')

    def all_on_callback(self):
        for i, ctrl in enumerate(self.controls):
            ctrl.setStyleSheet(BTN_GREEN); ctrl.setChecked(True)
            self.send_relay_command(f"relay off {self.get_relay_id(i)}")
        self.send_relay_command('close all')

    def send_relay_command(self, command):
        if self.Numato_port and self.Numato_port.is_open:
            try:
                self.Numato_port.write(f"{command}\r".encode('utf-8'))
                time.sleep(0.005)
            except serial.SerialException:
                QMessageBox.critical(self, 'Error', 'Failed to communicate with the device')

    # ── droplet slots ─────────────────────────
    def Characterize_Droplet(self, input, chemostat_number=1,
                              purge_duration=0, flow_duration=0, drive_duration=0):
        self.Characterize_Droplet_thread = DropletWorker(
            "characterize", input,
            purge_duration   = float(self.purge_duration_Input.text()),
            flow_duration    = float(self.flow_duration_Input.text()),
            drive_duration   = float(self.drive_duration_Input.text()),
            chemostat_number = int(self.inlet_Input.text()))
        self.Characterize_Droplet_thread.start()

    def PWM_droplet(self, input_1=12, input_2=13):
        self.PWM_thread = DropletWorker(
            "PWM", input_1, input_2,
            PWM_duration1    = float(self.PWM_duration1_Input.text()),
            PWM_duration2    = float(self.PWM_duration2_Input.text()),
            PWM_totalduration = float(self.PWM_totalduration_Input.text()))
        self.PWM_thread.start()

    def alter_Arduino_state(self, checked):
        if checked:
            self.TurnOnVolts.setText('Volts ON')
            self.TurnOnVolts.setStyleSheet(BTN_GREEN)
            send_to_arduino(2)
        else:
            self.TurnOnVolts.setText('Voltage Signal')
            self.TurnOnVolts.setStyleSheet(BTN_RED)
            send_to_arduino(3)

    # ── timelapse slots ───────────────────────
    def read_Exposure_values(self):
        self.selected_exposures = []
        for row in range(self.Exposures_table.rowCount()):
            filter = self.Exposures_table.cellWidget(row, 0).value()
            exposure = self.Exposures_table.cellWidget(row, 1).value()
            if exposure > 0:
                self.selected_exposures.append([filter, exposure])
        self.interval = int(self.cycle_Interval_Input.text())
        self.cycles   = int(self.cycle_Input.text())
        print("Exposures:", self.selected_exposures, "| Interval:", self.interval, "| Experiment duration:", self.cycles)

    def create_centered_checkbox(self):
        frame  = QFrame()
        layout = QHBoxLayout(frame)
        layout.setContentsMargins(0, 0, 0, 0)
        layout.setAlignment(Qt.AlignCenter)
        cb = QCheckBox(frame); cb.setEnabled(False)
        layout.addWidget(cb)
        return frame

    def add_loading_step(self):
        selected_inputs = [sb.value() for sb in self.Chemostat_inputs if sb.value() > 0]
        ring_status     = [cb.isChecked() for cb in self.Chemostats]
        if not selected_inputs and not any(ring_status):
            return
        step = {
            "input1": selected_inputs[0] if len(selected_inputs) > 0 else "",
            "input2": selected_inputs[1] if len(selected_inputs) > 1 else "",
            "rings":  ring_status
        }
        self.Chemostat_protocol_steps.append(step)
        col = self.current_protocol_table_step
        i1 = QTableWidgetItem(str(step["input1"])); i1.setTextAlignment(Qt.AlignCenter)
        i2 = QTableWidgetItem(str(step["input2"])); i2.setTextAlignment(Qt.AlignCenter)
        self.Chemostat_protocol_table.setItem(0, col, i1)
        self.Chemostat_protocol_table.setItem(1, col, i2)
        for i, is_on in enumerate(step["rings"]):
            cb = QCheckBox(); cb.setChecked(is_on); cb.setEnabled(False)
            w  = QWidget(); lyt = QVBoxLayout(w)
            lyt.setContentsMargins(0, 0, 0, 0)
            lyt.setAlignment(cb, Qt.AlignCenter)
            lyt.addWidget(cb)
            self.Chemostat_protocol_table.setCellWidget(2 + i, col, w)
        self.current_protocol_table_step += 1

    def clear_all_steps(self):
        self.Chemostat_protocol_steps = []
        self.current_protocol_table_step = 0
        for c in range(self.Chemostat_protocol_table.columnCount()):
            for r in range(self.Chemostat_protocol_table.rowCount()):
                if r < 2:
                    self.Chemostat_protocol_table.setItem(r, c, QTableWidgetItem(""))
                else:
                    self.Chemostat_protocol_table.setCellWidget(r, c, self.create_centered_checkbox())

    def clear_last_step(self):
        if not self.Chemostat_protocol_steps:
            return
        self.Chemostat_protocol_steps.pop()
        self.current_protocol_table_step = max(0, self.current_protocol_table_step - 1)
        col = self.current_protocol_table_step
        for r in range(self.Chemostat_protocol_table.rowCount()):
            if r < 2:
                self.Chemostat_protocol_table.setItem(r, col, QTableWidgetItem(""))
            else:
                frame = self.Chemostat_protocol_table.cellWidget(r, col)
                if frame:
                    cb = frame.layout().itemAt(0).widget()
                    if cb: cb.setChecked(False)

    def export_to_csv(self):
        if not self.Chemostat_protocol_steps:
            return
        fp, _ = QFileDialog.getSaveFileName(self, "Save Sequence to CSV", "",
                                             "CSV Files (*.csv);;All Files (*)")
        if not fp:
            return
        with open(fp, mode="w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["Step","Input 1","Input 2"] + [f"Ring {i+1}" for i in range(8)])
            for idx, step in enumerate(self.Chemostat_protocol_steps, 1):
                w.writerow([f"Step {idx}", step["input1"], step["input2"]]
                           + ["ON" if s else "OFF" for s in step["rings"]])

    def browse_folder(self):
        folder = QFileDialog.getExistingDirectory(self, "Select Folder")
        if folder:
            os.chdir(folder)
            self.directory_input.setText(folder)
    
    def _sam2_status(self, msg: str):
        """Update the SAM2 status label and print to console."""
        print(f"[SAM2] {msg}")
        if hasattr(self, "sam2_status_label"):
            self.sam2_status_label.setText(f"Status: {msg}")
            QApplication.processEvents()


    # ── 1. SAM2 tab builder ─────────────────────────────────────────────────────
    def _build_sam2_panel(self, parent_layout):
        """Build the SAM2 controls tab (called from __init__)."""
        info = QLabel(
            "① Capture one initial BF image per position (pre-experiment).\n"
            "② Annotate each image with SAM2 (live point-click preview).\n"
            "③ Start the SAM2-tracked timelapse.\n\n"
            f"Chemostat fires when a droplet shrinks below "
            f"{SAM2Config.AREA_THRESHOLD * 100:.0f}% of its initial area.\n"
            "Each position tracks its own loop counter independently.\n"
            "Use the Timing panel to set imaging interval and total duration."
        )
        info.setStyleSheet("color:#aaa; font-size:12px;")
        info.setWordWrap(True)
        parent_layout.addWidget(info)

        buttons = [
            ("① Capture initial BF for all positions",     BTN_BLUE,
            lambda: self.capture_bf_for_all_positions()),
            ("② Annotate all positions with SAM2",         BTN_BLUE,
            lambda: self.annotate_all_positions_sam2()),
            ("③ Start SAM2-tracked timelapse experiment",  BTN_GREEN,
            lambda: self.TimeLapse_Experiment_SAM2(
                interval_min=int(self.cycle_Interval_Input.text()),
                total_min=int(self.cycle_Input.text()),
            )),
        ]
        for label, style, slot in buttons:
            btn = QPushButton(label)
            btn.setStyleSheet(style)
            btn.setMinimumHeight(36)
            btn.clicked.connect(slot)
            parent_layout.addWidget(btn)

        self.sam2_status_label = QLabel("Status: idle")
        self.sam2_status_label.setStyleSheet("color:#ffcc44; font-size:11px;")
        parent_layout.addWidget(self.sam2_status_label)
        parent_layout.addStretch()


    # ── 2. Step ①: capture initial BF images ───────────────────────────────────
    def capture_bf_for_all_positions(self):
        """
        For every saved stage position:
        • Move the stage to that position.
        • Snap one BF image.
        • Save as  <base>/Position_<N>/BF/PosN_initialBF.tiff
        Paths are stored in sam2_mgr for the annotation step.
        """
        if not self.positions:
            QMessageBox.warning(self, "No positions",
                                "Save at least one stage position first.")
            return

        base = self.directory_input.text().strip() or "."
        self.sam2_mgr.base_dir = base

        self.DIALamp_activate()

        for idx, (px, py, pz) in enumerate(self.positions):
            pos_num = idx + 1
            bf_dir  = os.path.join(base, f"Position_{pos_num}", "BF")
            os.makedirs(bf_dir, exist_ok=True)

            self._sam2_status(f"Moving to position {pos_num} …")
            self.mmc.setXYPosition(px, py)
            self.mmc.setPosition(pz)
            self.mmc.waitForSystem()
            time.sleep(0.6)

            self._sam2_status(f"Snapping initial BF for position {pos_num} …")
            
            raw_arr = self.snap_DIA_image()

            save_path = os.path.join(bf_dir, f"Pos{pos_num}_initialBF.tiff")
            tiff.imwrite(save_path, raw_arr)

            # Initialise per-position state
            self.sam2_mgr.ref_image_path[idx] = save_path
            self.sam2_mgr.loop_bf_paths[idx]  = []
            self.sam2_mgr.pos_loop_count[idx] = 0

            self._sam2_status(f"Pos {pos_num}: saved → {save_path}")

        self._sam2_status("Initial BF capture complete for all positions.")
        self.DIALamp_deactivate()
        QMessageBox.information(
            self, "Done",
            "Initial BF images captured for all positions.\n"
            "Proceed to Step ② to annotate.",
        )


    # ── 3. Step ②: SAM2 annotation ─────────────────────────────────────────────
    def annotate_all_positions_sam2(self):
        """
        For each position, open the initial BF image in a matplotlib window.
        The user annotates the droplet; the mask is saved as:
            <base>/Position_<N>/BF/PosN_initialMask.tiff
        and stored in sam2_mgr.ref_mask[idx].
        """
        if not self.sam2_mgr.ref_image_path:
            QMessageBox.warning(self, "No BF images",
                                "Capture initial BF images first (Step ①).")
            return

        self.sam2_mgr.load_predictor()

        for idx in sorted(self.sam2_mgr.ref_image_path):
            pos_num  = idx + 1
            img_path = self.sam2_mgr.ref_image_path[idx]
            bf_dir   = os.path.dirname(img_path)

            self._sam2_status(
                f"Annotate Pos {pos_num} – click droplet, press Enter to accept …"
            )

            gray = SAM2Manager.load_gray_uint8(img_path)

            # Build a single-frame SAM2 state purely for the annotation window
            tmp_dir = os.path.join(
                self.sam2_mgr.base_dir, f"_sam2_annotate_tmp_pos{idx}"
            )
            SAM2Manager.write_sam2_frame_folder([img_path], tmp_dir)
            state = self.sam2_mgr.predictor.init_state(video_path=tmp_dir)

            mask = self.sam2_mgr.annotate_frame_interactive(gray, state)
            self.sam2_mgr.ref_mask[idx]    = mask
            self.sam2_mgr.initial_area[idx] = int(mask.sum())

            mask_path = os.path.join(bf_dir, f"Pos{pos_num}_initialMask.tiff")
            tiff.imwrite(mask_path, mask.astype(np.uint8))

            self._sam2_status(
                f"Pos {pos_num} annotated – initial area = {mask.sum()} px"
            )

        self._sam2_status("All positions annotated. Ready for Step ③.")
        QMessageBox.information(self, "Done",
                                "All positions annotated. Ready to start.")


    # ── 4. Steps 3-6: SAM2-tracked timelapse ────────────────────────────────────
    def TimeLapse_Experiment_SAM2(self, interval_min: int, total_min: int):
        """
        Time-driven timelapse with per-position SAM2 tracking.

        Parameters
        ----------
        interval_min : int  – minutes between imaging rounds (e.g. 3)
        total_min    : int  – total experiment duration in minutes (e.g. 120)

        Loop structure
        ──────────────
        Every `interval_min` minutes:
        [Step 3]  BF-image ALL positions and measure droplet area via SAM2.
                    SAM2 context for position P = [initial_BF] +
                    [all BF images taken in P's *current* loop so far].
                    Append the new BF to P's current-loop context.
                    Save the new BF image and its SAM2 mask.

        [Step 4]  Identify positions where area < AREA_THRESHOLD × initial.

        [Step 5]  FL imaging of triggered positions only.

        [Step 6]  Chemostat protocol for triggered positions only.
                    After chemostat:
                    • Reset that position's loop-BF list (context → initial only).
                    • Increment that position's loop counter.
        """
        if not self.sam2_mgr.ref_mask:
            QMessageBox.warning(self, "Not annotated",
                                "Complete Steps ① and ② before starting.")
            return
        if not self.positions:
            QMessageBox.warning(self, "No positions", "No stage positions saved.")
            return
        
        if interval_min <= 0 or total_min <= 0:
            QMessageBox.warning(self, "Invalid timing",
                                "Set a non-zero interval and total duration before starting.")
            return

        base = self.directory_input.text().strip() or "."
        self.sam2_mgr.base_dir = base

        purge_dur    = float(self.purge_duration_Input.text())
        flow_dur     = float(self.flow_duration_Input.text())
        drive_dur    = float(self.drive_duration_Input.text())
        interval_sec = interval_min * 60
        total_sec    = total_min    * 60

        # ── Initialise valve state (identical to original TimeLapse_Experiment) ──
        init_states = [
            (0, False), (1, True),  (2, False), (3, True),  (4, True),  (5, False),
            (6, False), (7, True),  (8, False), (9, False),  (10, False), (11, False),
            (12, True), (13, True), (14, True), (15, True),
        ]
        for v, s in init_states:
            self.control_valve(v, state=s)

        experiment_start = time.time()
        tick_number      = 0   # counts how many imaging rounds have been done

        self._sam2_status(
            f"Experiment started – interval {interval_min} min / "
            f"total {total_min} min"
        )
        print(f"\n{'='*62}")
        print(f"  SAM2 TIMELAPSE  |  interval {interval_min} min  "
            f"|  total {total_min} min")
        print(f"{'='*62}")

        # ── outer timing loop ────────────────────────────────────────────────────
        while True:
            now               = time.time()
            elapsed_sec       = now - experiment_start

            # Stop when total time is reached
            if elapsed_sec >= total_sec:
                break

            # Sleep until the next tick is due
            next_due_sec = tick_number * interval_sec
            wait_for     = next_due_sec - elapsed_sec
            if wait_for > 0:
                self._sam2_status(
                    f"Waiting {wait_for:.0f} s until next imaging round …"
                )
                time.sleep(wait_for)

            # ── start of one imaging round ───────────────────────────────────────
            tick_number         += 1
            tick_wall_start      = time.time()
            elapsed_min          = (tick_wall_start - experiment_start) / 60.0

            # Round the elapsed time to the nearest interval for the filename tag
            # e.g. at 3.02 min with interval=3  →  "3min"
            time_tag_min  = round(elapsed_min / interval_min) * interval_min
            time_tag_str  = f"{time_tag_min:.0f}min"

            print(f"\n{'─'*62}")
            print(f"  Tick {tick_number}  |  ~{time_tag_min:.0f} min elapsed")
            print(f"{'─'*62}")

            triggered_positions = []   # positions whose area fell below threshold

            self.DIALamp_activate()

            # ── Step 3: BF image every position, measure area ────────────────────
            for idx, (px, py, pz) in enumerate(self.positions):
                pos_num  = idx + 1
                # pos_loop is the 1-based loop label shown in filenames
                pos_loop = self.sam2_mgr.pos_loop_count.get(idx, 0) + 1
                initial  = self.sam2_mgr.initial_area.get(idx, 0)

                bf_dir = os.path.join(base, f"Position_{pos_num}", "BF")
                os.makedirs(bf_dir, exist_ok=True)

                # e.g.  Pos1_Loop3_BF_6min.tiff
                bf_fname = f"Pos{pos_num}_Loop{pos_loop}_BF_{time_tag_str}.tiff"
                bf_path  = os.path.join(bf_dir, bf_fname)

                self._sam2_status(
                    f"BF snap – Pos {pos_num}  Loop {pos_loop}  @ {time_tag_str}"
                )
                self.mmc.setXYPosition(px, py)
                self.mmc.setPosition(pz)
                self.mmc.waitForSystem()
                time.sleep(0.5)

                raw_arr = self.snap_DIA_image()
                tiff.imwrite(bf_path, raw_arr)

                # ── SAM2 area measurement ─────────────────────────────────────────
                if idx not in self.sam2_mgr.ref_mask:
                    print(f"  Pos {pos_num}: no reference mask, skipping SAM2.")
                    continue

                mask, area = self.sam2_mgr.measure_area_with_context(idx, bf_path)

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                # Save the SAM2 mask for this time-point
                # e.g.  Pos1_Loop3_mask_6min.tiff
                mask_fname = (
                    f"Pos{pos_num}_Loop{pos_loop}_mask_{time_tag_str}.tiff"
                )
                tiff.imwrite(os.path.join(bf_dir, mask_fname),
                            mask.astype(np.uint8))

                # Append this BF to the current-loop context for future ticks
                self.sam2_mgr.append_loop_bf(idx, bf_path)

                ratio = area / initial if initial > 0 else 1.0
                print(
                    f"  Pos {pos_num}  Loop {pos_loop}: "
                    f"area = {area} px  ({ratio * 100:.1f}% of initial  "
                    f"[threshold {SAM2Config.AREA_THRESHOLD * 100:.0f}%])"
                )

                if initial > 0 and area < SAM2Config.AREA_THRESHOLD * initial:
                    triggered_positions.append(idx)
                    self._sam2_status(
                        f"⚠ Pos {pos_num} TRIGGERED "
                        f"({ratio * 100:.1f}% < "
                        f"{SAM2Config.AREA_THRESHOLD * 100:.0f}%)"
                    )

            self.DIALamp_deactivate()
            
            # ── Step 4: report triggered positions ──────────────────────────────
            if not triggered_positions:
                print("  ✓ All droplets within acceptable size.")
                self._sam2_status(
                    f"Tick {tick_number}: all areas OK. "
                    f"Next tick in ~{interval_min} min."
                )
                continue   # no FL imaging or chemostat needed this round

            names = [f"Pos {i + 1}" for i in triggered_positions]
            print(f"\n  ⚠ Triggered this tick: {', '.join(names)}")

            # ── Step 5: FL imaging of triggered positions only ───────────────────
            if self.selected_exposures:
                for idx in triggered_positions:
                    pos_num  = idx + 1
                    pos_loop = self.sam2_mgr.pos_loop_count.get(idx, 0) + 1

                    fl_dir = os.path.join(base, f"Position_{pos_num}", "FL")
                    os.makedirs(fl_dir, exist_ok=True)

                    self._sam2_status(
                        f"FL imaging – Pos {pos_num}  Loop {pos_loop}"
                    )
                    px, py, pz = self.positions[idx]
                    self.mmc.setXYPosition(px, py)
                    self.mmc.setPosition(pz)
                    self.mmc.waitForSystem()
                    time.sleep(0.5)

                    for filt, exp in self.selected_exposures:
                        raw_fl = self.snap_EPI_image(filt, exp)
                        # e.g.  Pos1_Loop3_filt1_100ms_6min.tiff
                        fl_fname = (
                            f"Pos{pos_num}_Loop{pos_loop}"
                            f"_filt{filt}_{exp}ms_{time_tag_str}.tiff"
                        )
                        tiff.imwrite(os.path.join(fl_dir, fl_fname), raw_fl)
                        time.sleep(0.2)

            # ── Step 6: chemostat protocol for triggered positions only ──────────
            if self.Chemostat_protocol_steps:
                self._sam2_status(
                    "Running chemostat for triggered positions …"
                )

                # Valve priming sequence (identical to original experiment)
                self.control_valve(15, state=False)
                time.sleep(10)
                self.control_valve(15, state=True)
                self.control_valve(7,  state=False)
                for v in [12, 13, 14, 15]:
                    self.control_valve(v, state=False)
                    time.sleep(5)
                    self.control_valve(v, state=True)
                self.control_valve(7, state=True)

                self.mmc.setProperty(self.DIAlamp, "State", 1)
                self.video_thread = VideoThread()
                self.video_thread.start()

                for step in self.Chemostat_protocol_steps:
                    i1    = step["input1"]
                    i2    = step["input2"]
                    rings = step["rings"]   # list[bool], one entry per position

                    # Purge inlet once per step (not once per position)
                    pw = DropletWorker("purge", i1, i2, purge_duration=purge_dur)
                    pw.start(); pw.wait()

                    for rn, active in enumerate(rings):
                        # Only act if the ring is checked AND this pos was triggered
                        if active and (rn in triggered_positions):
                            px, py, pz = self.positions[rn]
                            self.mmc.setXYPosition(px, py)
                            self.mmc.setPosition(pz)
                            self.mmc.waitForSystem()
                            time.sleep(0.3)

                            gw = DropletWorker(
                                "generate", i1, flow_duration=flow_dur
                            )
                            gw.start(); gw.wait()

                            self.video_thread.start_recording()
                            dw = DropletWorker(
                                "drive", i1,
                                drive_duration=drive_dur,
                                chemostat_number=rn + 1,
                            )
                            dw.start(); dw.wait()
                            self.video_thread.stop_recording()

                    # Close / re-open flush valves between steps
                    self.control_valve(15, state=False)
                    self.control_valve(7,  state=False)
                    time.sleep(5)
                    self.control_valve(15, state=True)
                    self.control_valve(7,  state=True)

                self.video_thread.stop()
                self.mmc.setProperty(self.DIAlamp, "State", 0)

            # ── Reset SAM2 context and advance loop counter per triggered pos ────
            for idx in triggered_positions:
                self.sam2_mgr.reset_loop_context(idx)
                self.sam2_mgr.pos_loop_count[idx] = (
                    self.sam2_mgr.pos_loop_count.get(idx, 0) + 1
                )
                new_loop = self.sam2_mgr.pos_loop_count[idx] + 1
                print(
                    f"  Pos {idx + 1}: SAM2 context reset → "
                    f"entering Loop {new_loop}"
                )

        # ── Experiment complete ──────────────────────────────────────────────────
        self.control_valve(0, state=True)
        print("\n=== SAM2 timelapse experiment complete ===")
        self._sam2_status("Experiment complete.")




    # ── timelapse experiment ──────────────────
    def TimeLapse_Experiment(self, num_loops, time_interval, positions_table,
                              selected_exposures, chemostat_protocol_table):
        init_states = [(0,False),(1,True),(2,False),(3,True),(4,True),(5,False),
                       (6,False),(7,True),(8,False),(9,False),(10,False),(11,False),
                       (12,True),(13,True),(14,True),(15,True)]
        for v, s in init_states:
            self.control_valve(v, state=s)

        purge_duration = float(self.purge_duration_Input.text())
        flow_duration  = float(self.flow_duration_Input.text())
        drive_duration = float(self.drive_duration_Input.text())

        for loop in range(num_loops):
            print(f"--- Loop {loop+1}/{num_loops} ---")
            loop_start = time.time()

            for ci in range(len(positions_table)):
                self.mmc.setXYPosition(positions_table[ci][0], positions_table[ci][1])
                self.mmc.setPosition(positions_table[ci][2])
                self.mmc.waitForSystem(); time.sleep(0.5)
                for filt, exp in selected_exposures:
                    img = self.snap_EPI_image(filt, exp)
                    fn  = f"Expt_{ci+1}_{filt}_{exp}_{loop+1}.tiff"
                    time.sleep(0.2)
                    img.save(os.path.join(".", fn))

            if self.Chemostat_protocol_steps:
                self.control_valve(15, state=False); time.sleep(30)
                self.control_valve(15, state=True)
                self.control_valve(7,  state=False)
                for v in [12, 13, 14, 15]:
                    self.control_valve(v, state=False); time.sleep(5)
                    self.control_valve(v, state=True)
                self.control_valve(7, state=True)
                self.mmc.setProperty(self.DIAlamp, 'State', 1)
                self.video_thread = VideoThread()
                self.video_thread.start()

                for step in self.Chemostat_protocol_steps:
                    i1 = step["input1"]; i2 = step["input2"]
                    pw = DropletWorker("purge", i1, i2, purge_duration=purge_duration)
                    pw.start(); pw.wait()
                    for rn, active in enumerate(step["rings"]):
                        if active:
                            self.mmc.setXYPosition(positions_table[rn][0], positions_table[rn][1])
                            self.mmc.setPosition(positions_table[rn][2])
                            gw = DropletWorker("generate", i1, flow_duration=flow_duration)
                            gw.start(); gw.wait()
                            self.video_thread.start_recording()
                            dw = DropletWorker("drive", i1, drive_duration=drive_duration, chemostat_number=rn+1)
                            dw.start(); dw.wait()
                            self.video_thread.stop_recording()
                    self.control_valve(15, state=False); self.control_valve(7, state=False)
                    time.sleep(5)
                    self.control_valve(15, state=True);  self.control_valve(7, state=True)

                self.video_thread.stop()
                self.mmc.setProperty(self.DIAlamp, 'State', 0)

            elapsed   = time.time() - loop_start
            remaining = time_interval * 60 - elapsed
            if remaining > 0:
                print(f"Waiting {remaining:.1f}s …")
                time.sleep(remaining)

        print("--- Experiment complete ---")
        self.control_valve(0, state=True)




    # ── 5. Dispatcher: replaces the original start-button connection ─────────────
    def _on_start_experiment(self):
        """
        If SAM2 reference masks have been annotated, use the SAM2-tracked
        experiment.  Otherwise fall back to the original timelapse.

        Wire up in __init__:
            self.start_experiment_button.clicked.connect(self._on_start_experiment)
        """
        if self.sam2_mgr.ref_mask:
            self.TimeLapse_Experiment_SAM2(
                interval_min=int(self.cycle_Interval_Input.text()),
                total_min=int(self.cycle_Input.text()),
            )
        else:
            self.TimeLapse_Experiment(
                num_loops=int(self.cycle_Input.text()),
                time_interval=int(self.cycle_Interval_Input.text()),
                positions_table=self.positions,
                selected_exposures=self.selected_exposures,
                chemostat_protocol_table=self.Chemostat_protocol_steps,
            )
            



    def closeEvent(self, event):
        self.all_on_callback()
        if self.Numato_port and self.Numato_port.is_open:
            self.Numato_port.close()
        arduino.close()
        event.accept()
# ─────────────────────────────────────────────
if __name__ == '__main__':
    app = QtWidgets.QApplication(sys.argv)
    app.setStyle('Fusion')
    app.setPalette(make_dark_palette())
    app.setStyleSheet(WIDGET_STYLE)
    window = MicroscopeControlGUI()
    sys.exit(app.exec_())



"""EXPERIMENT LOGIC OVERVIEW
─────────────────────────
Pre-experiment (manual, before clicking Start):
①  capture_bf_for_all_positions()   – snap & save one BF per position
②  annotate_all_positions_sam2()    – interactive SAM2 point annotation

Experiment (TimeLapse_Experiment_SAM2):
• Every `interval_min` minutes, BF-image ALL positions.
• Each position keeps its own independent loop counter.
• SAM2 context for position P at time T in loop L =
        [initial_BF]  +  [all BF images taken in loop L before time T]
    → the initial BF is always frame 0; current-loop BFs accumulate on top.
• If area < 70 % of initial  →  position is "triggered":
        – FL imaging for that position (saved to Position_N/FL/)
        – Chemostat protocol for that position
        – Current-loop BF list reset (context back to initial only)
        – That position's loop counter incremented by 1
• Experiment ends when total wall-clock time is reached.

FOLDER / FILE NAMING
─────────────────────
<base>/
Position_1/
    BF/
    Pos1_initialBF.tiff
    Pos1_initialMask.tiff
    Pos1_Loop1_BF_3min.tiff        ← first tick, loop 1
    Pos1_Loop1_mask_3min.tiff
    Pos1_Loop1_BF_6min.tiff        ← triggered here
    Pos1_Loop1_mask_6min.tiff
    Pos1_Loop2_BF_3min.tiff        ← context reset, loop 2 begins
    Pos1_Loop2_mask_3min.tiff
    …
    FL/
    Pos1_Loop1_filt1_100ms_6min.tiff
    Pos1_Loop2_filt2_50ms_21min.tiff
    …
Position_2/
    BF/  …
    FL/  …
"""

Connected to COM4 at 115200 baud.
Handshake successful!
Serial connection established.
Numato relay correctly connected
[SAM2] Moving to position 1 …
[SAM2] Snapping initial BF for position 1 …
[SAM2] Pos 1: saved → .\Position_1\BF\Pos1_initialBF.tiff
[SAM2] Moving to position 2 …
[SAM2] Snapping initial BF for position 2 …
[SAM2] Pos 2: saved → .\Position_2\BF\Pos2_initialBF.tiff
[SAM2] Moving to position 3 …
[SAM2] Snapping initial BF for position 3 …
[SAM2] Pos 3: saved → .\Position_3\BF\Pos3_initialBF.tiff
[SAM2] Initial BF capture complete for all positions.
[SAM2] Loading predictor …


INFO:root:Loaded checkpoint sucessfully


[SAM2] Predictor ready.
[SAM2] Annotate Pos 1 – click droplet, press Enter to accept …


frame loading (JPEG): 100%|██████████| 1/1 [00:00<00:00, 26.39it/s]
DEBUG:matplotlib.pyplot:Loaded backend QtAgg version 5.15.10.
DEBUG:matplotlib.font_manager:findfont: Matching sans\-serif:style=normal:variant=normal:weight=normal:stretch=normal:size=10.0.
DEBUG:matplotlib.font_manager:findfont: score(FontEntry(fname='c:\\ProgramData\\anaconda3\\envs\\chemostat_sam2\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\ttf\\cmex10.ttf', name='cmex10', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 10.05
DEBUG:matplotlib.font_manager:findfont: score(FontEntry(fname='c:\\ProgramData\\anaconda3\\envs\\chemostat_sam2\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\ttf\\DejaVuSans.ttf', name='DejaVu Sans', style='normal', variant='normal', weight=400, stretch='normal', size='scalable')) = 0.05
DEBUG:matplotlib.font_manager:findfont: score(FontEntry(fname='c:\\ProgramData\\anaconda3\\envs\\chemostat_sam2\\Lib\\site-packages\\matplotlib\\mpl-data\\fonts\\t

RuntimeError: No annotation given – add at least one positive point.

...

[SAM2] Annotate Pos 1 – click droplet, press Enter to accept …


frame loading (JPEG): 100%|██████████| 1/1 [00:00<00:00, 26.39it/s]


RuntimeError: No annotation given – add at least one positive point.

SystemExit: 0

c:\ProgramData\anaconda3\envs\chemostat_sam2\Lib\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
